In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Define paths
bureau_path = '../../data/raw/bureau.csv'
if not os.path.exists(bureau_path):
    bureau_path = '/Users/nguyenminhtri/FinalYearPro/data/raw/bureau.csv'

# 2. Load raw dataset
print(f"🔄 Loading raw bureau dataset from: {bureau_path}")
df_bureau = pd.read_csv(bureau_path)

print(f"✅ Initial shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")
print(f"🔑 Unique clients (SK_ID_CURR): {df_bureau['SK_ID_CURR'].nunique():,}")

🔄 Loading raw bureau dataset from: ../../data/raw/bureau.csv
✅ Initial shape: 1,716,428 rows | 17 columns
🔑 Unique clients (SK_ID_CURR): 305,811


In [2]:
# 1. Create binary indicator flags (0 / 1)
df_bureau['BUREAU_IS_CLOSED'] = (df_bureau['CREDIT_ACTIVE'] == 'Closed').astype(int)
df_bureau['BUREAU_IS_ACTIVE'] = (df_bureau['CREDIT_ACTIVE'] == 'Active').astype(int)
df_bureau['BUREAU_IS_SOLD'] = (df_bureau['CREDIT_ACTIVE'] == 'Sold').astype(int)
df_bureau['BUREAU_IS_BAD_DEBT'] = (df_bureau['CREDIT_ACTIVE'] == 'Bad debt').astype(int)

# 2. Severe risk combined flag (Bad debt or Sold to 3rd party)
df_bureau['BUREAU_IS_BAD_OR_SOLD'] = df_bureau['CREDIT_ACTIVE'].isin(['Sold', 'Bad debt']).astype(int)

# 3. Drop original string column to prevent redundancy in aggregation
df_bureau.drop(columns=['CREDIT_ACTIVE'], inplace=True)

print("✅ Successfully processed CREDIT_ACTIVE! Created 5 binary flags and dropped original column.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed CREDIT_ACTIVE! Created 5 binary flags and dropped original column.
📊 Current dataset shape: 1,716,428 rows | 21 columns


## 📌 Step 1: Processing `CREDIT_ACTIVE` (Loan Status)

### 💡 Justification & Strategy:
- **Data Quality:** Missing rate is `0.00%`.
- **Distribution:** `Closed` (62.88%), `Active` (36.74%), `Sold` (0.38%), `Bad debt` (0.001%).
- **Transformation:** Since `CREDIT_ACTIVE` is a string categorical column, it cannot be aggregated directly in a 1-to-Many relational structure. We encode it into binary indicator flags ($0/1$). This allows `groupby('SK_ID_CURR')` to calculate both the total count (`sum`) and proportion (`mean`) of each loan status for every client.
- **Cleanup:** The original string column `CREDIT_ACTIVE` is dropped post-encoding as it is no longer required for numeric aggregation.

In [3]:
# --- CLEANING & PRE-AGGREGATION: DAYS_CREDIT ---

# 1. Convert negative days to positive years
df_bureau['BUREAU_CREDIT_YEARS'] = df_bureau['DAYS_CREDIT'].abs() / 365.25

# 2. Recency Flags (Loans opened within 1 year / 6 months)
df_bureau['BUREAU_IS_RECENT_1Y'] = (df_bureau['DAYS_CREDIT'] >= -365).astype(int)
df_bureau['BUREAU_IS_RECENT_6M'] = (df_bureau['DAYS_CREDIT'] >= -180).astype(int)

print("✅ Successfully processed DAYS_CREDIT! Created BUREAU_CREDIT_YEARS, BUREAU_IS_RECENT_1Y, and BUREAU_IS_RECENT_6M.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed DAYS_CREDIT! Created BUREAU_CREDIT_YEARS, BUREAU_IS_RECENT_1Y, and BUREAU_IS_RECENT_6M.
📊 Current dataset shape: 1,716,428 rows | 24 columns


## 📌 Step 2: Processing `DAYS_CREDIT` (Loan Open Date Relative to Application)

### 💡 Justification & Strategy:
- **Data Quality:** Missing rate is `0.00%`. No structural anomalies/outliers observed ($Min = -2,922$ days, $Max = 0$ days).
- **Domain Rationale:** Recent credit applications signal potential liquidity strain or credit-seeking behavior.
- **Transformation:**
  1. Convert negative days into positive year units (`BUREAU_CREDIT_YEARS`).
  2. Create recency indicator flags for loans opened within the last 1 year (`BUREAU_IS_RECENT_1Y`) and 6 months (`BUREAU_IS_RECENT_6M`).

In [4]:
# --- CLEANING & PRE-AGGREGATION: CREDIT_DAY_OVERDUE ---

# 1. General Overdue Flag (> 0 days)
df_bureau['BUREAU_IS_OVERDUE'] = (df_bureau['CREDIT_DAY_OVERDUE'] > 0).astype(int)

# 2. Regulatory Delinquency Buckets (Groups 2 to 5)
df_bureau['BUREAU_DPD_GR2'] = ((df_bureau['CREDIT_DAY_OVERDUE'] >= 10) & (df_bureau['CREDIT_DAY_OVERDUE'] <= 90)).astype(int)
df_bureau['BUREAU_DPD_GR3'] = ((df_bureau['CREDIT_DAY_OVERDUE'] >= 91) & (df_bureau['CREDIT_DAY_OVERDUE'] <= 180)).astype(int)
df_bureau['BUREAU_DPD_GR4'] = ((df_bureau['CREDIT_DAY_OVERDUE'] >= 181) & (df_bureau['CREDIT_DAY_OVERDUE'] <= 360)).astype(int)
df_bureau['BUREAU_DPD_GR5'] = (df_bureau['CREDIT_DAY_OVERDUE'] > 360).astype(int)

# 3. Formal Non-Performing Loan (NPL / Bad Debt Flag = Groups 3 + 4 + 5)
df_bureau['BUREAU_IS_NPL'] = (df_bureau['CREDIT_DAY_OVERDUE'] >= 91).astype(int)

print("✅ Successfully processed CREDIT_DAY_OVERDUE with Banking Standards (Groups 2 to 5)!")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed CREDIT_DAY_OVERDUE with Banking Standards (Groups 2 to 5)!
📊 Current dataset shape: 1,716,428 rows | 30 columns


## 📌 Step 3: Processing `CREDIT_DAY_OVERDUE` (Days Past Due - DPD)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `0.00%`. Heavily zero-inflated ($>99\%$ values are `0`).
- **Domain Benchmark (Regulatory Delinquency Buckets - Groups 2 to 5):**
  Short delays ($DPD < 10$ days) are ignored as operational/technical noise. We construct explicit binary indicators starting from Group 2 (Special Mention) up to Group 5 (Loss):
  1. **Group 2 - Special Mention ($10 \le DPD \le 90$ days):** Early warning stage indicating material credit risk.
  2. **Group 3 - Substandard ($91 \le DPD \le 180$ days):** Formal Non-Performing Loan (NPL / Bad Debt entry).
  3. **Group 4 - Doubtful ($181 \le DPD \le 360$ days):** Severe delinquency with high expected loss.
  4. **Group 5 - Loss ($DPD > 360$ days):** Chronic bad debt with near-total capital loss.
- **Transformation:**
  1. Retain cleaned continuous DPD feature for `max` (peak historical delinquency) and `mean` aggregation.
  2. Create an overall overdue flag (`BUREAU_IS_OVERDUE` for $DPD > 0$).
  3. Construct non-overlapping regulatory delinquency indicators (`BUREAU_DPD_GR2`, `BUREAU_DPD_GR3`, `BUREAU_DPD_GR4`, `BUREAU_DPD_GR5`).
  4. Create an explicit Non-Performing Loan flag (`BUREAU_IS_NPL` for $DPD \ge 91$, encompassing Groups 3, 4, and 5).

In [5]:
# --- STEP 0: DROP UNINFORMATIVE / REDUNDANT COLUMNS ---

cols_to_drop = ['CREDIT_CURRENCY', 'AMT_ANNUITY']

# Filter only existing columns to avoid errors if re-run
existing_cols_to_drop = [col for col in cols_to_drop if col in df_bureau.columns]

if existing_cols_to_drop:
    df_bureau.drop(columns=existing_cols_to_drop, inplace=True)
    print(f"🗑️ Successfully dropped uninformative columns: {existing_cols_to_drop}")

print(f"📊 Current dataset shape after Step 0: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

🗑️ Successfully dropped uninformative columns: ['CREDIT_CURRENCY', 'AMT_ANNUITY']
📊 Current dataset shape after Step 0: 1,716,428 rows | 28 columns


## 🧹 Step 0: Initial Feature Selection & Uninformative Column Removal

### 💡 Justification & Strategy:
Prior to column-by-column preprocessing, two raw columns from `bureau.csv` are dropped to optimize memory and eliminate noise:

1. **`CREDIT_CURRENCY` (Zero Variance):** Over `99.9%` of historical records share the exact same currency class (`currency 1`). Features with near-zero variance contribute no explanatory power to tree-based split algorithms.
2. **`AMT_ANNUITY` (High Missingness & Data Redundancy):** Suffers from an extremely high missing rate ($>70\%$) because external financial institutions do not consistently report monthly installment amounts to the Credit Bureau. Monthly payment obligation is comprehensively captured in `application_train.csv` and `previous_application.csv`.

In [6]:
# --- CLEANING & PRE-AGGREGATION: DAYS_CREDIT_ENDDATE ---

# 1. Flag structural anomalies (> 10 years in future OR < 10 years in past)
anom_mask = (df_bureau['DAYS_CREDIT_ENDDATE'] > 3652.5) | (df_bureau['DAYS_CREDIT_ENDDATE'] < -3652.5)
df_bureau['BUREAU_ENDDATE_ANOM'] = anom_mask.astype(int)

# 2. Clean feature by replacing anomalies with NaN
df_bureau['DAYS_CREDIT_ENDDATE_CLEAN'] = df_bureau['DAYS_CREDIT_ENDDATE'].copy()
df_bureau.loc[anom_mask, 'DAYS_CREDIT_ENDDATE_CLEAN'] = np.nan

# 3. Derive total loan contract duration (in days)
df_bureau['BUREAU_LOAN_DURATION_DAYS'] = df_bureau['DAYS_CREDIT_ENDDATE_CLEAN'] - df_bureau['DAYS_CREDIT']

# 4. Binary future obligation flags
df_bureau['BUREAU_IS_FUTURE_OBLIGATION'] = (df_bureau['DAYS_CREDIT_ENDDATE_CLEAN'] > 0).astype(int)
df_bureau['BUREAU_IS_LONG_TERM'] = (df_bureau['DAYS_CREDIT_ENDDATE_CLEAN'] > 365).astype(int)

print("✅ Successfully cleaned and processed DAYS_CREDIT_ENDDATE!")
print(f"   Anomalies cleaned: {anom_mask.sum():,} rows replaced with NaN.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully cleaned and processed DAYS_CREDIT_ENDDATE!
   Anomalies cleaned: 74,134 rows replaced with NaN.
📊 Current dataset shape: 1,716,428 rows | 33 columns


## 📌 Step 4: Processing `DAYS_CREDIT_ENDDATE` (Remaining Days to Contract Maturity)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `6.15%`. Contains severe data entry anomalies with extreme max ($31,199$ days $\approx 85$ years) and extreme min ($-42,060$ days $\approx -115$ years).
- **Interpretation:**
  - Negative values ($< 0$): Loans that already matured in the past.
  - Positive values ($> 0$): Active future financial obligations.
- **Cleaning & Transformation Strategy:**
  1. Flag severe anomalies ($|DAYS\_CREDIT\_ENDDATE| > 3652.5$ days / 10 years) into `BUREAU_ENDDATE_ANOM` and replace anomalous values with `NaN`.
  2. Derive contract duration in days (`BUREAU_LOAN_DURATION_DAYS` = `DAYS_CREDIT_ENDDATE_CLEAN - DAYS_CREDIT`).
  3. Construct future financial burden indicators: `BUREAU_IS_FUTURE_OBLIGATION` ($> 0$ days) and `BUREAU_IS_LONG_TERM` ($> 365$ days).

In [7]:
# --- CLEANING & PRE-AGGREGATION: DAYS_ENDDATE_FACT ---

# 1. Flag structural anomalies (< -10 years in the past)
anom_fact_mask = df_bureau['DAYS_ENDDATE_FACT'] < -3652.5
df_bureau['BUREAU_FACT_ANOM'] = anom_fact_mask.astype(int)

# 2. Clean feature by replacing anomalies with NaN
df_bureau['DAYS_ENDDATE_FACT_CLEAN'] = df_bureau['DAYS_ENDDATE_FACT'].copy()
df_bureau.loc[anom_fact_mask, 'DAYS_ENDDATE_FACT_CLEAN'] = np.nan

# 3. Calculate difference between actual settlement and contractual end date (in days)
if 'DAYS_CREDIT_ENDDATE_CLEAN' in df_bureau.columns:
    df_bureau['BUREAU_SETTLEMENT_DIFF_DAYS'] = df_bureau['DAYS_ENDDATE_FACT_CLEAN'] - df_bureau['DAYS_CREDIT_ENDDATE_CLEAN']

    # 4. Behavioral flags (Early Prepayment vs Delayed Settlement)
    df_bureau['BUREAU_IS_EARLY_SETTLED'] = (df_bureau['BUREAU_SETTLEMENT_DIFF_DAYS'] < 0).astype(int)
    df_bureau['BUREAU_IS_LATE_SETTLED'] = (df_bureau['BUREAU_SETTLEMENT_DIFF_DAYS'] > 0).astype(int)

print("✅ Successfully cleaned and processed DAYS_ENDDATE_FACT!")
print(f"   Anomalies cleaned: {anom_fact_mask.sum():,} rows replaced with NaN.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully cleaned and processed DAYS_ENDDATE_FACT!
   Anomalies cleaned: 1 rows replaced with NaN.
📊 Current dataset shape: 1,716,428 rows | 38 columns


## 📌 Step 5: Processing `DAYS_ENDDATE_FACT` (Actual Loan Settlement Date)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `36.92%`, which structurally aligns with active loans that have not yet reached maturity. Contains extreme negative anomalies ($Min = -42,023$ days $\approx -115$ years).
- **Domain Rationale:** Comparing the actual settlement date (`DAYS_ENDDATE_FACT`) against the contractual end date (`DAYS_CREDIT_ENDDATE`) reveals borrower payment behavior—distinguishing early prepayments from delayed contract closures.
- **Cleaning & Transformation Strategy:**
  1. Flag extreme historical anomalies ($DAYS\_ENDDATE\_FACT < -3652.5$ days / 10 years) into `BUREAU_FACT_ANOM` and clean anomalous entries to `NaN`.
  2. Compute settlement delay/prepayment gap in days (`BUREAU_SETTLEMENT_DIFF_DAYS` = `DAYS_ENDDATE_FACT_CLEAN - DAYS_CREDIT_ENDDATE_CLEAN`).
  3. Construct behavior flags for early prepayment (`BUREAU_IS_EARLY_SETTLED`) and delayed settlement (`BUREAU_IS_LATE_SETTLED`).

In [8]:
# --- CLEANING & PRE-AGGREGATION: AMT_CREDIT_MAX_OVERDUE ---

# 1. Flag missing status before imputation
df_bureau['BUREAU_MAX_OVERDUE_IS_NA'] = df_bureau['AMT_CREDIT_MAX_OVERDUE'].isna().astype(int)

# 2. Impute missing values with 0.0 for numeric safety
df_bureau['AMT_CREDIT_MAX_OVERDUE_CLEAN'] = df_bureau['AMT_CREDIT_MAX_OVERDUE'].fillna(0.0)

# 3. Binary flag indicating historical overdue occurrence
df_bureau['BUREAU_HAS_MAX_OVERDUE'] = (df_bureau['AMT_CREDIT_MAX_OVERDUE_CLEAN'] > 0).astype(int)

print("✅ Successfully processed AMT_CREDIT_MAX_OVERDUE!")
print(f"   Imputed {df_bureau['BUREAU_MAX_OVERDUE_IS_NA'].sum():,} missing values with 0.0.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed AMT_CREDIT_MAX_OVERDUE!
   Imputed 1,124,488 missing values with 0.0.
📊 Current dataset shape: 1,716,428 rows | 41 columns


## 📌 Step 6: Processing `AMT_CREDIT_MAX_OVERDUE` (Max Historical Overdue Amount)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Exhibits a high missing rate of `65.51%` (only 591,940 valid observations).
- **Domain Rationale:** Missing entries in external credit bureau reports indicate loans with no recorded historical default or overdue payment history (structural non-delinquency).
- **Transformation Strategy:**
  1. Construct an explicit missing indicator flag (`BUREAU_MAX_OVERDUE_IS_NA`) to preserve missingness signal.
  2. Impute missing values with `0.0` into `AMT_CREDIT_MAX_OVERDUE_CLEAN` for numeric aggregation.
  3. Derive a binary occurrence flag (`BUREAU_HAS_MAX_OVERDUE`) indicating historical delinquency.

In [9]:
# --- CLEANING & PRE-AGGREGATION: CNT_CREDIT_PROLONG ---

# 1. Any Prolongation Flag
df_bureau['BUREAU_IS_PROLONGED'] = (df_bureau['CNT_CREDIT_PROLONG'] > 0).astype(int)

# 2. Repeated Prolongation Flag (>= 2 times)
df_bureau['BUREAU_IS_MULTIPLE_PROLONGED'] = (df_bureau['CNT_CREDIT_PROLONG'] >= 2).astype(int)

print("✅ Successfully processed CNT_CREDIT_PROLONG!")
print(f"   Prolonged loans found: {df_bureau['BUREAU_IS_PROLONGED'].sum():,} records.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed CNT_CREDIT_PROLONG!
   Prolonged loans found: 9,114 records.
📊 Current dataset shape: 1,716,428 rows | 43 columns


## 📌 Step 7: Processing `CNT_CREDIT_PROLONG` (Credit Prolongation Count)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `0.00%`. Heavily zero-inflated ($>99.4\%$ values are `0`).
- **Domain Rationale:** Loan extensions or prolongations reflect borrower liquidity distress and inability to meet original repayment schedules. Multiple prolongations ($\ge 2$) serve as a strong early warning indicator for structural credit risk.
- **Transformation Strategy:**
  1. Retain original discrete feature to calculate `sum` and `max` during aggregation.
  2. Construct a binary prolongation occurrence flag (`BUREAU_IS_PROLONGED` for $CNT\_CREDIT\_PROLONG > 0$).
  3. Construct a high-risk repeated prolongation flag (`BUREAU_IS_MULTIPLE_PROLONGED` for $CNT\_CREDIT\_PROLONG \ge 2$).

In [10]:
# --- CLEANING & PRE-AGGREGATION: AMT_CREDIT_SUM (IN-PLACE) ---

# 1. Direct in-place imputation (No extra memory copy)
df_bureau['AMT_CREDIT_SUM'] = df_bureau['AMT_CREDIT_SUM'].fillna(0.0)

# 2. Zero credit limit indicator flag
df_bureau['BUREAU_CREDIT_SUM_IS_ZERO'] = (df_bureau['AMT_CREDIT_SUM'] == 0.0).astype(int)

print("✅ Successfully cleaned AMT_CREDIT_SUM in-place!")
print(f"   Zero credit limit records found: {df_bureau['BUREAU_CREDIT_SUM_IS_ZERO'].sum():,} rows.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully cleaned AMT_CREDIT_SUM in-place!
   Zero credit limit records found: 66,595 rows.
📊 Current dataset shape: 1,716,428 rows | 44 columns


## 📌 Step 9: Processing `AMT_CREDIT_SUM` (Total Credit Limit)

### 💡 Justification & Strategy:
- **Data Quality:** Negligible missing rate ($0.00\%$, exactly 13 missing rows out of $1,716,428$). Valid minimum value ($Min = 0.00$). Heavily right-skewed distribution.
- **In-Place Cleaning:** Since missing values are negligible, we directly overwrite `AMT_CREDIT_SUM` in-place to avoid memory duplication and redundant `_CLEAN` columns.
- **Transformation Strategy:**
  1. Impute 13 missing values with `0.0` directly into `AMT_CREDIT_SUM`.
  2. Construct a zero-limit indicator flag (`BUREAU_CREDIT_SUM_IS_ZERO` for $AMT\_CREDIT\_SUM == 0$).

In [11]:
# --- CLEANING & PRE-AGGREGATION: AMT_CREDIT_SUM_DEBT (IN-PLACE) ---

# 1. Missing Flag before Imputation
df_bureau['BUREAU_DEBT_IS_NA'] = df_bureau['AMT_CREDIT_SUM_DEBT'].isna().astype(int)

# 2. In-place Imputation & Negative Value Clipping
df_bureau['AMT_CREDIT_SUM_DEBT'] = df_bureau['AMT_CREDIT_SUM_DEBT'].fillna(0.0).clip(lower=0.0)

# 3. Cross-Column Feature Engineering: Credit Utilization Ratio per Loan
df_bureau['BUREAU_DEBT_RATIO'] = df_bureau['AMT_CREDIT_SUM_DEBT'] / (df_bureau['AMT_CREDIT_SUM'] + 1e-5)

# 4. Zero Debt Flag
df_bureau['BUREAU_IS_DEBT_FREE'] = (df_bureau['AMT_CREDIT_SUM_DEBT'] == 0.0).astype(int)

print("✅ Successfully processed AMT_CREDIT_SUM_DEBT & created BUREAU_DEBT_RATIO in-place!")
print(f"   Imputed {df_bureau['BUREAU_DEBT_IS_NA'].sum():,} missing rows with 0.0.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed AMT_CREDIT_SUM_DEBT & created BUREAU_DEBT_RATIO in-place!
   Imputed 257,669 missing rows with 0.0.
📊 Current dataset shape: 1,716,428 rows | 47 columns


## 📌 Step 10: Processing `AMT_CREDIT_SUM_DEBT` (Current Active Debt Amount)

### 💡 Justification & Domain Engineering Strategy:
- **Difference from `AMT_CREDIT_SUM`:** `AMT_CREDIT_SUM` represents the total approved credit limit/initial loan principal, whereas `AMT_CREDIT_SUM_DEBT` captures the actual remaining unpaid debt balance at the application time.
- **Data Quality:** Contains missing values ($\sim 15-16\%$) primarily from closed/unreported loans, along with minor negative accounting artifacts.
- **In-Place Cleaning & Feature Engineering Strategy:**
  1. Construct a missingness indicator flag (`BUREAU_DEBT_IS_NA`).
  2. Impute missing entries with `0.0` and clip negative accounting anomalies to `0.0` directly in-place on `AMT_CREDIT_SUM_DEBT`.
  3. Derive the **Credit Utilization Ratio** (`BUREAU_DEBT_RATIO` = `AMT_CREDIT_SUM_DEBT / (AMT_CREDIT_SUM + 1e-5)`), measuring credit leverage exposure.

In [12]:
# --- CLEANING & PRE-AGGREGATION: AMT_CREDIT_SUM_LIMIT (IN-PLACE) ---

# 1. Missing Flag before Imputation (34.48% missing)
df_bureau['BUREAU_LIMIT_IS_NA'] = df_bureau['AMT_CREDIT_SUM_LIMIT'].isna().astype(int)

# 2. Extract Over-Limit Signal from negative values before cleaning
df_bureau['BUREAU_IS_OVER_LIMIT'] = (df_bureau['AMT_CREDIT_SUM_LIMIT'] < 0.0).astype(int)

# 3. Direct In-place Imputation & Negative Clipping
df_bureau['AMT_CREDIT_SUM_LIMIT'] = df_bureau['AMT_CREDIT_SUM_LIMIT'].fillna(0.0).clip(lower=0.0)

# 4. Positive Available Buffer Flag
df_bureau['BUREAU_HAS_AVAILABLE_LIMIT'] = (df_bureau['AMT_CREDIT_SUM_LIMIT'] > 0.0).astype(int)

print("✅ Successfully processed AMT_CREDIT_SUM_LIMIT in-place!")
print(f"   Over-limit credit cards identified: {df_bureau['BUREAU_IS_OVER_LIMIT'].sum():,} records.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed AMT_CREDIT_SUM_LIMIT in-place!
   Over-limit credit cards identified: 351 records.
📊 Current dataset shape: 1,716,428 rows | 50 columns


## 📌 Step 11: Processing `AMT_CREDIT_SUM_LIMIT` (Available Credit Card Limit)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** High missing rate ($34.48\%$), structurally corresponding to non-revolving credit products (Consumer loans, Mortgages). Contains negative values ($Min = -586,406.11$).
- **Domain Rationale (Over-limit Behavior):** A negative available credit limit ($< 0$) signifies that the credit card holder has spent beyond their credit ceiling or incurred penalties causing account over-limit status—a severe financial distress signal.
- **In-Place Cleaning Strategy:**
  1. Capture the over-limit signal into `BUREAU_IS_OVER_LIMIT` before cleaning negative numbers.
  2. Construct a missingness flag (`BUREAU_LIMIT_IS_NA`) for non-revolving loans.
  3. Impute `NaN` with `0.0` and clip negative values to `0.0` directly in-place on `AMT_CREDIT_SUM_LIMIT`.
  4. Derive available credit buffer flag (`BUREAU_HAS_AVAILABLE_LIMIT`).

In [13]:
# --- CLEANING & PRE-AGGREGATION: AMT_CREDIT_SUM_OVERDUE (IN-PLACE) ---

# 1. Direct In-place Clipping (Lower bound 0.0)
df_bureau['AMT_CREDIT_SUM_OVERDUE'] = df_bureau['AMT_CREDIT_SUM_OVERDUE'].clip(lower=0.0)

# 2. Binary flag indicating active overdue balance
df_bureau['BUREAU_HAS_SUM_OVERDUE'] = (df_bureau['AMT_CREDIT_SUM_OVERDUE'] > 0.0).astype(int)

# 3. Cross-Column Feature Engineering: Overdue-to-Debt Ratio
if 'AMT_CREDIT_SUM_DEBT' in df_bureau.columns:
    df_bureau['BUREAU_OVERDUE_TO_DEBT_RATIO'] = (
        df_bureau['AMT_CREDIT_SUM_OVERDUE'] / (df_bureau['AMT_CREDIT_SUM_DEBT'] + 1e-5)
    )

print("✅ Successfully processed AMT_CREDIT_SUM_OVERDUE in-place!")
print(f"   Active overdue records found: {df_bureau['BUREAU_HAS_SUM_OVERDUE'].sum():,} rows.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed AMT_CREDIT_SUM_OVERDUE in-place!
   Active overdue records found: 4,158 rows.
📊 Current dataset shape: 1,716,428 rows | 52 columns


## 📌 Step 12: Processing `AMT_CREDIT_SUM_OVERDUE` (Current Overdue Amount)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `0.00%`. Min is `0.00` with no negative artifacts. Heavily zero-inflated ($>99\%$ zero values). Max reaches $3,756,681.00$.
- **Domain Rationale:** Active overdue balances outside the financial institution represent an immediate credit risk signal at application time.
- **In-Place Cleaning & Feature Engineering Strategy:**
  1. Apply lower-bound clipping at `0.0` directly in-place on `AMT_CREDIT_SUM_OVERDUE` for numeric integrity.
  2. Derive an active current overdue indicator flag (`BUREAU_HAS_SUM_OVERDUE` for $AMT\_CREDIT\_SUM\_OVERDUE > 0$).
  3. Compute the **Overdue-to-Debt Ratio** (`BUREAU_OVERDUE_TO_DEBT_RATIO` = `AMT_CREDIT_SUM_OVERDUE / (AMT_CREDIT_SUM_DEBT + 1e-5)`), capturing the proportion of active debt that has gone default.

In [14]:
# --- CLEANING & PRE-AGGREGATION: CREDIT_TYPE (CATEGORICAL OHE) ---

# 1. Define top 5 core risk categories
top_credit_types = [
    'Consumer credit',
    'Credit card',
    'Car loan',
    'Mortgage',
    'Microloan'
]

# 2. Map rare long-tail categories to 'Other'
df_bureau['CREDIT_TYPE_CLEAN'] = df_bureau['CREDIT_TYPE'].apply(
    lambda x: x if x in top_credit_types else 'Other'
)

# 3. Perform One-Hot Encoding (OHE)
type_dummies = pd.get_dummies(df_bureau['CREDIT_TYPE_CLEAN'], prefix='BUREAU_TYPE').astype(int)
df_bureau = pd.concat([df_bureau, type_dummies], axis=1)

# 4. Drop original string column & temporary column
df_bureau.drop(columns=['CREDIT_TYPE', 'CREDIT_TYPE_CLEAN'], inplace=True)

print("✅ Successfully One-Hot Encoded CREDIT_TYPE!")
print(f"   Created 6 binary risk indicators: {list(type_dummies.columns)}")
print(f"   Dropped original string column 'CREDIT_TYPE'.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully One-Hot Encoded CREDIT_TYPE!
   Created 6 binary risk indicators: ['BUREAU_TYPE_Car loan', 'BUREAU_TYPE_Consumer credit', 'BUREAU_TYPE_Credit card', 'BUREAU_TYPE_Microloan', 'BUREAU_TYPE_Mortgage', 'BUREAU_TYPE_Other']
   Dropped original string column 'CREDIT_TYPE'.
📊 Current dataset shape: 1,716,428 rows | 57 columns


## 📌 Step 13: Processing `CREDIT_TYPE` (Loan Category & Risk Profiling)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `0.00%` across 15 distinct categories. Heavily long-tailed distribution where top 2 categories (`Consumer credit` & `Credit card`) account for `96.35%` of all records.
- **Domain Rationale:** Loan types carry fundamentally distinct default risk profiles:
  - `Microloans` ($0.72\%$): High-risk indicators associated with liquidity distress.
  - `Mortgages` & `Car loans`: Asset-backed secured credits with lower default probabilities.
  - `Credit cards`: Revolving operational credit limits.
- **Transformation Strategy:**
  1. Consolidate the 10 ultra-rare categories ($< 0.25\%$ combined) into a single `'Other'` class to eliminate feature sparsity.
  2. Perform One-Hot Encoding (OHE) on the top 5 distinct types plus `'Other'`.
  3. Drop the original categorical string column `CREDIT_TYPE` in-place to free up memory.

In [15]:
# --- CLEANING & PRE-AGGREGATION: DAYS_CREDIT_UPDATE (IN-PLACE) ---

# 1. Flag structural anomalies (extreme past < -10 years OR future > 0 days)
anom_update_mask = (df_bureau['DAYS_CREDIT_UPDATE'] < -3652.5) | (df_bureau['DAYS_CREDIT_UPDATE'] > 0)
df_bureau['BUREAU_UPDATE_ANOM'] = anom_update_mask.astype(int)

# 2. Direct In-place Anomaly Cleaning (replace with NaN)
df_bureau.loc[anom_update_mask, 'DAYS_CREDIT_UPDATE'] = np.nan

# 3. Information Recency / Freshness Flags
df_bureau['BUREAU_UPDATED_RECENT_30D'] = (df_bureau['DAYS_CREDIT_UPDATE'] >= -30).astype(int)
df_bureau['BUREAU_UPDATED_RECENT_90D'] = (df_bureau['DAYS_CREDIT_UPDATE'] >= -90).astype(int)

print("✅ Successfully processed DAYS_CREDIT_UPDATE in-place!")
print(f"   Anomalies cleaned: {anom_update_mask.sum():,} rows replaced with NaN.")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully processed DAYS_CREDIT_UPDATE in-place!
   Anomalies cleaned: 112 rows replaced with NaN.
📊 Current dataset shape: 1,716,428 rows | 60 columns


## 📌 Step 14: Processing `DAYS_CREDIT_UPDATE` (Information Recency / Freshness)

### 💡 Justification & Domain Engineering Strategy:
- **Data Quality:** Missing rate is `0.00%`. Contains extreme past anomalies ($Min = -41,947$ days $\approx -115$ years) and future anomalies ($Max = 372$ days $> 0$).
- **Domain Rationale:** The recency of credit bureau updates reflects data freshness and active tracking by external lenders. Recent updates ($\le 30$ or $90$ days) ensure the reported balances and default statuses are up to date.
- **In-Place Cleaning & Feature Engineering Strategy:**
  1. Flag structural time anomalies ($DAYS\_CREDIT\_UPDATE < -3652.5$ OR $> 0$) into `BUREAU_UPDATE_ANOM`.
  2. Impute anomalous entries with `np.nan` directly in-place on `DAYS_CREDIT_UPDATE` for calculation accuracy.
  3. Derive information recency flags: `BUREAU_UPDATED_RECENT_30D` ($\ge -30$ days) and `BUREAU_UPDATED_RECENT_90D` ($\ge -90$ days).

In [16]:
# --- CHECK CURRENT COLUMNS IN DF_BUREAU ---

print(f"📊 Total columns in df_bureau: {len(df_bureau.columns)}\n")
print("📋 List of current column names:")
print("=" * 60)
for i, col in enumerate(df_bureau.columns, 1):
    print(f"{i:02d}. {col:<35} | dtype: {df_bureau[col].dtype}")
print("=" * 60)

📊 Total columns in df_bureau: 60

📋 List of current column names:
01. SK_ID_CURR                          | dtype: int64
02. SK_ID_BUREAU                        | dtype: int64
03. DAYS_CREDIT                         | dtype: int64
04. CREDIT_DAY_OVERDUE                  | dtype: int64
05. DAYS_CREDIT_ENDDATE                 | dtype: float64
06. DAYS_ENDDATE_FACT                   | dtype: float64
07. AMT_CREDIT_MAX_OVERDUE              | dtype: float64
08. CNT_CREDIT_PROLONG                  | dtype: int64
09. AMT_CREDIT_SUM                      | dtype: float64
10. AMT_CREDIT_SUM_DEBT                 | dtype: float64
11. AMT_CREDIT_SUM_LIMIT                | dtype: float64
12. AMT_CREDIT_SUM_OVERDUE              | dtype: float64
13. DAYS_CREDIT_UPDATE                  | dtype: float64
14. BUREAU_IS_CLOSED                    | dtype: int64
15. BUREAU_IS_ACTIVE                    | dtype: int64
16. BUREAU_IS_SOLD                      | dtype: int64
17. BUREAU_IS_BAD_DEBT                

In [17]:
# --- STEP 15A: DROP RAW ANOMALY COLUMNS ---

raw_cols_to_remove = ['DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE']
existing_raw_to_drop = [col for col in raw_cols_to_remove if col in df_bureau.columns]

if existing_raw_to_drop:
    df_bureau.drop(columns=existing_raw_to_drop, inplace=True)
    print(f"🗑️ Successfully dropped raw anomaly columns: {existing_raw_to_drop}")
else:
    print("ℹ️ Raw anomaly columns were already removed or not found.")

print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

🗑️ Successfully dropped raw anomaly columns: ['DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE']
📊 Current dataset shape: 1,716,428 rows | 57 columns


## 📌 Step 15A: Cleanup Old Raw Anomaly Columns

### 💡 Justification & Strategy:
- Remove raw columns (`DAYS_CREDIT_ENDDATE`, `DAYS_ENDDATE_FACT`, `AMT_CREDIT_MAX_OVERDUE`) that contain uncleaned structural anomalies and extreme outliers.
- Ensures only sanitized features (`_CLEAN`) are passed forward into the aggregation stage.

In [18]:
# --- STEP 15B: ADD FINAL FINANCIAL RATIOS ---

# 1. Total principal amount actually paid off (clipped at 0.0)
df_bureau['BUREAU_CREDIT_PAID_AMOUNT'] = (
    df_bureau['AMT_CREDIT_SUM'] - df_bureau['AMT_CREDIT_SUM_DEBT']
).clip(lower=0.0)

# 2. Repayment Completion Ratio
df_bureau['BUREAU_PAID_RATIO'] = df_bureau['BUREAU_CREDIT_PAID_AMOUNT'] / (df_bureau['AMT_CREDIT_SUM'] + 1e-5)

print("✅ Successfully created BUREAU_CREDIT_PAID_AMOUNT & BUREAU_PAID_RATIO!")
print(f"📊 Current dataset shape: {df_bureau.shape[0]:,} rows | {df_bureau.shape[1]} columns")

✅ Successfully created BUREAU_CREDIT_PAID_AMOUNT & BUREAU_PAID_RATIO!
📊 Current dataset shape: 1,716,428 rows | 59 columns


## 📌 Step 15B: Adding High-Value Financial Features

### 💡 Justification & Strategy:
- Derive principal paid amount (`BUREAU_CREDIT_PAID_AMOUNT`) by subtracting current debt from initial approved credit.
- Compute repayment completion ratio (`BUREAU_PAID_RATIO`) to capture borrower reliability in clearing debt obligations.

In [19]:
# --- CHECK CURRENT COLUMNS IN DF_BUREAU ---

print(f"📊 Total columns in df_bureau: {len(df_bureau.columns)}\n")
print("📋 List of current column names:")
print("=" * 60)
for i, col in enumerate(df_bureau.columns, 1):
    print(f"{i:02d}. {col:<35} | dtype: {df_bureau[col].dtype}")
print("=" * 60)

📊 Total columns in df_bureau: 59

📋 List of current column names:
01. SK_ID_CURR                          | dtype: int64
02. SK_ID_BUREAU                        | dtype: int64
03. DAYS_CREDIT                         | dtype: int64
04. CREDIT_DAY_OVERDUE                  | dtype: int64
05. CNT_CREDIT_PROLONG                  | dtype: int64
06. AMT_CREDIT_SUM                      | dtype: float64
07. AMT_CREDIT_SUM_DEBT                 | dtype: float64
08. AMT_CREDIT_SUM_LIMIT                | dtype: float64
09. AMT_CREDIT_SUM_OVERDUE              | dtype: float64
10. DAYS_CREDIT_UPDATE                  | dtype: float64
11. BUREAU_IS_CLOSED                    | dtype: int64
12. BUREAU_IS_ACTIVE                    | dtype: int64
13. BUREAU_IS_SOLD                      | dtype: int64
14. BUREAU_IS_BAD_DEBT                  | dtype: int64
15. BUREAU_IS_BAD_OR_SOLD               | dtype: int64
16. BUREAU_CREDIT_YEARS                 | dtype: float64
17. BUREAU_IS_RECENT_1Y                 | 

In [20]:
# --- STEP 1: RENAME ALL REMAINING COLUMNS TO HAVE 'BUREAU_' PREFIX ---

rename_dict = {
    'DAYS_CREDIT': 'BUREAU_DAYS_CREDIT',
    'DAYS_CREDIT_UPDATE': 'BUREAU_DAYS_CREDIT_UPDATE',
    'DAYS_CREDIT_ENDDATE_CLEAN': 'BUREAU_DAYS_CREDIT_ENDDATE_CLEAN',
    'DAYS_ENDDATE_FACT_CLEAN': 'BUREAU_DAYS_ENDDATE_FACT_CLEAN',
    'AMT_CREDIT_SUM': 'BUREAU_AMT_CREDIT_SUM',
    'AMT_CREDIT_SUM_DEBT': 'BUREAU_AMT_CREDIT_SUM_DEBT',
    'AMT_CREDIT_SUM_LIMIT': 'BUREAU_AMT_CREDIT_SUM_LIMIT',
    'AMT_CREDIT_SUM_OVERDUE': 'BUREAU_AMT_CREDIT_SUM_OVERDUE',
    'AMT_CREDIT_MAX_OVERDUE_CLEAN': 'BUREAU_AMT_CREDIT_MAX_OVERDUE_CLEAN',
    'CREDIT_DAY_OVERDUE': 'BUREAU_CREDIT_DAY_OVERDUE',
    'CNT_CREDIT_PROLONG': 'BUREAU_CNT_CREDIT_PROLONG'
}

# Đổi tên trực tiếp
df_bureau.rename(columns=rename_dict, inplace=True)

print("✅ Đã đổi tên 11 cột còn thiếu tiền tố BUREAU_ thành công!")

✅ Đã đổi tên 11 cột còn thiếu tiền tố BUREAU_ thành công!


In [21]:
# --- STEP 2: AGGREGATION DICTIONARY VỚI TÊN CỘT ĐÃ ĐỒNG NHẤT BUREAU_ ---

bureau_agg_dict = {
    # 1. Các cột mốc thời gian (Days Features)
    'BUREAU_DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
    'BUREAU_DAYS_CREDIT_UPDATE': ['max', 'mean'],
    'BUREAU_DAYS_CREDIT_ENDDATE_CLEAN': ['min', 'max', 'mean'],
    'BUREAU_DAYS_ENDDATE_FACT_CLEAN': ['min', 'max', 'mean'],
    'BUREAU_CREDIT_YEARS': ['max', 'mean'],
    'BUREAU_LOAN_DURATION_DAYS': ['max', 'mean'],
    'BUREAU_SETTLEMENT_DIFF_DAYS': ['max', 'mean'],

    # 2. Các cột quy mô số tiền (Monetary Amounts)
    'BUREAU_AMT_CREDIT_SUM': ['sum', 'mean', 'max'],
    'BUREAU_AMT_CREDIT_SUM_DEBT': ['sum', 'mean', 'max'],
    'BUREAU_AMT_CREDIT_SUM_LIMIT': ['sum', 'mean', 'max'],
    'BUREAU_AMT_CREDIT_SUM_OVERDUE': ['sum', 'mean', 'max'],
    'BUREAU_AMT_CREDIT_MAX_OVERDUE_CLEAN': ['max', 'mean'],
    'BUREAU_CREDIT_PAID_AMOUNT': ['sum', 'mean', 'max'],

    # 3. Các chỉ số tỷ lệ & Đếm ngày trễ hạn (Ratios & Counts)
    'BUREAU_CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'BUREAU_CNT_CREDIT_PROLONG': ['sum', 'max'],
    'BUREAU_DEBT_RATIO': ['max', 'mean'],
    'BUREAU_OVERDUE_TO_DEBT_RATIO': ['max', 'mean'],
    'BUREAU_PAID_RATIO': ['max', 'mean'],

    # 4. Các cờ nhị phân (Binary Flags & One-Hot Encoded Features) -> CHỈ LẤY sum VÀ mean
    'BUREAU_IS_CLOSED': ['sum', 'mean'],
    'BUREAU_IS_ACTIVE': ['sum', 'mean'],
    'BUREAU_IS_SOLD': ['sum', 'mean'],
    'BUREAU_IS_BAD_DEBT': ['sum', 'mean'],
    'BUREAU_IS_BAD_OR_SOLD': ['sum', 'mean'],
    'BUREAU_IS_RECENT_1Y': ['sum', 'mean'],
    'BUREAU_IS_RECENT_6M': ['sum', 'mean'],
    'BUREAU_IS_OVERDUE': ['sum', 'mean'],
    'BUREAU_DPD_GR2': ['sum', 'mean'],
    'BUREAU_DPD_GR3': ['sum', 'mean'],
    'BUREAU_DPD_GR4': ['sum', 'mean'],
    'BUREAU_DPD_GR5': ['sum', 'mean'],
    'BUREAU_IS_NPL': ['sum', 'mean'],
    'BUREAU_ENDDATE_ANOM': ['sum', 'mean'],
    'BUREAU_IS_FUTURE_OBLIGATION': ['sum', 'mean'],
    'BUREAU_IS_LONG_TERM': ['sum', 'mean'],
    'BUREAU_FACT_ANOM': ['sum', 'mean'],
    'BUREAU_IS_EARLY_SETTLED': ['sum', 'mean'],
    'BUREAU_IS_LATE_SETTLED': ['sum', 'mean'],
    'BUREAU_MAX_OVERDUE_IS_NA': ['sum', 'mean'],
    'BUREAU_HAS_MAX_OVERDUE': ['sum', 'mean'],
    'BUREAU_IS_PROLONGED': ['sum', 'mean'],
    'BUREAU_IS_MULTIPLE_PROLONGED': ['sum', 'mean'],
    'BUREAU_CREDIT_SUM_IS_ZERO': ['sum', 'mean'],
    'BUREAU_DEBT_IS_NA': ['sum', 'mean'],
    'BUREAU_IS_DEBT_FREE': ['sum', 'mean'],
    'BUREAU_LIMIT_IS_NA': ['sum', 'mean'],
    'BUREAU_IS_OVER_LIMIT': ['sum', 'mean'],
    'BUREAU_HAS_AVAILABLE_LIMIT': ['sum', 'mean'],
    'BUREAU_HAS_SUM_OVERDUE': ['sum', 'mean'],
    'BUREAU_TYPE_Car loan': ['sum', 'mean'],
    'BUREAU_TYPE_Consumer credit': ['sum', 'mean'],
    'BUREAU_TYPE_Credit card': ['sum', 'mean'],
    'BUREAU_TYPE_Microloan': ['sum', 'mean'],
    'BUREAU_TYPE_Mortgage': ['sum', 'mean'],
    'BUREAU_TYPE_Other': ['sum', 'mean'],
    'BUREAU_UPDATE_ANOM': ['sum', 'mean'],
    'BUREAU_UPDATED_RECENT_30D': ['sum', 'mean'],
    'BUREAU_UPDATED_RECENT_90D': ['sum', 'mean']
}

print(f"✅ Đã khai báo xong Dictionary đồng nhất cho {len(bureau_agg_dict)} cột!")

✅ Đã khai báo xong Dictionary đồng nhất cho 57 cột!


In [22]:
import pandas as pd

print("🚀 Đang tiến hành Groupby nén dữ liệu...")

# Chỉ lọc lấy các cột thực sự có trong dataframe
valid_aggs = {col: aggs for col, aggs in bureau_agg_dict.items() if col in df_bureau.columns}

# 1. OVERALL AGGREGATION
bureau_agg = df_bureau.groupby('SK_ID_CURR').agg(valid_aggs)
bureau_agg.columns = [f"{col}_{stat.upper()}" for col, stat in bureau_agg.columns]
bureau_agg['BUREAU_TOTAL_LOAN_COUNT'] = df_bureau.groupby('SK_ID_CURR').size()

# 2. ACTIVE LOANS AGGREGATION
if 'BUREAU_IS_ACTIVE' in df_bureau.columns:
    active_mask = df_bureau['BUREAU_IS_ACTIVE'] == 1
    active_agg = df_bureau[active_mask].groupby('SK_ID_CURR').agg(valid_aggs)
    # Thay tiền tố BUREAU_ thành BUREAU_ACTIVE_ cho rõ phân khúc
    active_agg.columns = [f"BUREAU_ACTIVE_{col[7:]}_{stat.upper()}" for col, stat in active_agg.columns]
    active_agg['BUREAU_ACTIVE_LOAN_COUNT'] = df_bureau[active_mask].groupby('SK_ID_CURR').size()

    bureau_agg = bureau_agg.join(active_agg, how='left')

# 3. CLOSED LOANS AGGREGATION
if 'BUREAU_IS_CLOSED' in df_bureau.columns:
    closed_mask = df_bureau['BUREAU_IS_CLOSED'] == 1
    closed_agg = df_bureau[closed_mask].groupby('SK_ID_CURR').agg(valid_aggs)
    # Thay tiền tố BUREAU_ thành BUREAU_CLOSED_ cho rõ phân khúc
    closed_agg.columns = [f"BUREAU_CLOSED_{col[7:]}_{stat.upper()}" for col, stat in closed_agg.columns]
    closed_agg['BUREAU_CLOSED_LOAN_COUNT'] = df_bureau[closed_mask].groupby('SK_ID_CURR').size()

    bureau_agg = bureau_agg.join(closed_agg, how='left')

bureau_agg.reset_index(inplace=True)

print("==================================================")
print("🎉 HOÀN THÀNH ÉP PHẲNG BẢNG BUREAU!")
print("==================================================")
print(f"📊 Kích thước bảng phẳng 'bureau_agg': {bureau_agg.shape[0]:,} khách hàng | {bureau_agg.shape[1]} thuộc tính")

🚀 Đang tiến hành Groupby nén dữ liệu...
🎉 HOÀN THÀNH ÉP PHẲNG BẢNG BUREAU!
📊 Kích thước bảng phẳng 'bureau_agg': 305,811 khách hàng | 373 thuộc tính


In [23]:
import os
import gc

# 1. Define output path & save parquet
output_path = '/data/processed/table/df_bureau_clean_fe.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
bureau_agg.to_parquet(output_path, index=False)

print("=" * 60)
print(f"📊 Final Dataset Shape: {bureau_agg.shape[0]:,} rows | {bureau_agg.shape[1]} columns")
print(f"💾 Saved successfully to: {output_path}")
print("=" * 60)

# 2. Cleanup RAM
del df_bureau, bureau_agg
gc.collect()
print("🧹 Memory cleared successfully!")

OSError: [Errno 30] Read-only file system: '/data'